# UQRoute-TC — four-model pilot audit

Run these two code cells in order in Colab. **No GPU, vLLM server, or new model inference is required.** The notebook reads the four saved five-case development pilots, checks they use the same IDs and valid evidence, and prints the official scorer's case-level results. It saves a compact audit JSON in Drive. Five cases measure feasibility, not full benchmark accuracy or a routing result.


## 1. Read and validate the four saved pilots


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import statistics
import subprocess

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC/pilots')
folders = {
    'Qwen 1.5B': 'qwen25-15b-dev-five-v1',
    'Llama 3B': 'llama32-3b-dev-five-v1',
    'Qwen 7B': 'qwen25-7b-dev-five-v1',
    'Qwen 14B AWQ': 'qwen25-14b-awq-feasibility-five-v1',
}

pilots = {}
expected_ids = None
for label, folder in folders.items():
    location = root / folder
    manifest = json.loads((location / 'manifest.json').read_text())
    records = [json.loads(line) for line in
               (location / 'pilot.predictions.jsonl').read_text().splitlines() if line.strip()]
    scores = json.loads((location / 'scores.json').read_text())
    case_ids = manifest['case_ids']
    assert len(case_ids) == 5 and len(set(case_ids)) == 5
    assert [record['id'] for record in records] == case_ids
    assert all(record['uqroute']['status'] == 'complete' and
               record['uqroute']['evidence']['valid'] for record in records)
    assert scores['overall']['overall']['n'] == 5
    if expected_ids is None:
        expected_ids = case_ids
    assert case_ids == expected_ids, f'{label} has different case IDs'
    pilots[label] = {'manifest': manifest, 'records': records, 'scores': scores}
    result = scores['overall']['overall']
    seconds = statistics.median(record['uqroute']['elapsed_seconds'] for record in records)
    print(f'{label:13} scored={result["correct"]}/5  '
          f'valid_evidence=5/5  median_case_seconds={seconds:.1f}  '
          f'code={manifest["code_revision"][:7]}')

print('Same five clean development IDs across all four models.')


## 2. Compare official case scores and save the audit


In [ ]:
import importlib.util
import sys

benchmark = Path('/content/robustbench-tc-release')
if not benchmark.exists():
    subprocess.run(['git', 'clone',
        'https://github.com/WillChow66/robustbench-tc-release.git',
        str(benchmark)], check=True)
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'],
                               text=True).strip() == benchmark_revision

scorer_path = benchmark / 'leaderboard/scoring/eval_results.py'
spec = importlib.util.spec_from_file_location('pinned_pilot_scorer', scorer_path)
scorer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(scorer)

labels = list(folders)
correct = {label: {record['id']: bool(scorer.score_record(record)['correct'])
                   for record in pilots[label]['records']} for label in labels}
print('Case ID'.ljust(51), *(label.rjust(13) for label in labels))
for case_id in expected_ids:
    print(case_id.ljust(51),
          *('correct'.rjust(13) if correct[label][case_id] else 'wrong'.rjust(13)
            for label in labels))

fallback = 'Qwen 14B AWQ'
rescue = {label: sum(not correct[label][case_id] and correct[fallback][case_id]
                     for case_id in expected_ids) for label in labels if label != fallback}
print('Five-case fallback rescues by small model:', rescue)
print('These five-case counts are descriptive; they do not validate a routing threshold.')

audit = {
    'benchmark_revision': benchmark_revision,
    'case_ids': expected_ids,
    'models': {
        label: {
            'model_id': pilots[label]['manifest']['model_id'],
            'model_revision': pilots[label]['manifest']['model_revision'],
            'code_revision': pilots[label]['manifest']['code_revision'],
            'overall': pilots[label]['scores']['overall']['overall'],
            'median_case_seconds': statistics.median(
                record['uqroute']['elapsed_seconds'] for record in pilots[label]['records']),
            'case_correct': correct[label],
            'valid_evidence': 5,
        } for label in labels
    },
    'five_case_fallback_rescues': rescue,
    'scope': 'same five clean development cases; feasibility only',
}
audit_path = root / 'four_model_pilot_audit.json'
audit_path.write_text(json.dumps(audit, indent=2) + '\n')
print('Audit saved:', audit_path)
